In [5]:
import random

alphabet_num = 10
length = 20 # 長さは固定
convert_length = 3

alphabet = tuple(chr(i + 65) for i in range(alphabet_num))

rule_num = 200

rules = []
while len(rules) < rule_num:
    before = ''.join(random.choices(alphabet, k=convert_length))
    after = ''.join(random.choices(alphabet, k=convert_length))
    if (before != after) and ((before, after) not in rules):
        rules.append((before, after))

print(rules)

[('BJJ', 'GCG'), ('BCH', 'GBJ'), ('CEB', 'IFB'), ('DDD', 'DIG'), ('BCI', 'JHB'), ('ADA', 'EJD'), ('IHE', 'JBJ'), ('IIB', 'CGI'), ('BGF', 'JBH'), ('EFH', 'IDA'), ('JIB', 'JJJ'), ('EIC', 'CGI'), ('EFJ', 'HEC'), ('CAG', 'IID'), ('HIG', 'JCB'), ('EIF', 'JAC'), ('CJD', 'JAG'), ('DDE', 'EEI'), ('IIE', 'JHG'), ('BEG', 'DGE'), ('EDG', 'EGA'), ('DGF', 'JBH'), ('HJE', 'FCG'), ('FFG', 'FDG'), ('GBI', 'CEB'), ('HBB', 'FHB'), ('BBD', 'JFD'), ('BJH', 'GCH'), ('AIC', 'BCB'), ('FFC', 'DJE'), ('FGA', 'EJC'), ('EFJ', 'CAH'), ('CCJ', 'GIB'), ('EFI', 'BCF'), ('HFA', 'JDC'), ('HED', 'JJF'), ('GCG', 'HBE'), ('FAF', 'DEA'), ('HAI', 'FIF'), ('BJC', 'IAI'), ('DJG', 'EDC'), ('CDB', 'FCH'), ('JDH', 'HHI'), ('IAC', 'HDA'), ('IDG', 'IHA'), ('HBG', 'GJI'), ('EFF', 'FFG'), ('JDD', 'IIC'), ('HCE', 'GBJ'), ('GEB', 'BFH'), ('JCI', 'GAI'), ('GIC', 'GDG'), ('GAC', 'HFD'), ('BEF', 'AAB'), ('AIC', 'EBB'), ('DGE', 'EEH'), ('IFI', 'BAJ'), ('JIF', 'ICD'), ('HAC', 'JCI'), ('JDA', 'GGJ'), ('BHE', 'HED'), ('BFJ', 'JHA'), ('EBJ',

In [6]:
def ngram(a, b, n):
    assert len(a) == len(b)
    return sum(1 for i in range(len(a) - n + 1) if a[i:i+n] == b[i:i+n])

def find_successors(s, rules):
    out = []
    for lhs, rhs in rules:
        k = len(lhs)
        start = 0
        while True:
            idx = s.find(lhs, start)
            if idx == -1:
                break
            nxt = s[:idx] + rhs + s[idx + k:]
            out.append((nxt, (idx, lhs, rhs)))
            start = idx + 1
    return out


In [57]:
import random

def sample_path(rules, alphabet, length=20, step_num=10, direction="forward", max_tries=1000):
    """
    ランダムに step_num ステップの確実なパスを生成して返す。
    direction が "backward" のときは規則を反転させて生成し、
    返り値は常に「前向き」の (before, after, path) にそろえる。
    """
    if direction not in ("forward", "backward"):
        raise ValueError("direction must be 'forward' or 'backward'")

    # 後ろ向きは規則を反転して前向きと同じロジックで辿る
    use_rules = rules if direction == "forward" else [(rhs, lhs) for (lhs, rhs) in rules]

    for _ in range(max_tries):
        seed = ''.join(random.choices(alphabet, k=length))
        state = seed
        path = [state]

        for _step in range(step_num):
            successors = find_successors(state, use_rules)  # [(next_str, (idx,lhs,rhs)), ...]
            if not successors:
                break
            nxt, _info = random.choice(successors)
            state = nxt
            path.append(state)

        # きっちり step_num ステップ進めたら採用
        if len(path) == step_num + 1:
            if direction == "forward":
                before, after, out_path = path[0], path[-1], path
            else:
                # 反転規則で得た seed -> state を前向きに読むと state -> seed
                before, after, out_path = path[-1], path[0], list(reversed(path))
            return before, after, out_path

    raise RuntimeError("Failed to sample a path of the requested length within max_tries.")

before, after, path = sample_path(rules, alphabet, length=20, step_num=10, direction="backward")
print(before, after)
print(path[0], "-> ... ->", path[-1]) 
print("path:", path)

CGEFIGIIGDBEGHICGAFF CFCEBGCHDABAEHCEJCFF
CGEFIGIIGDBEGHICGAFF -> ... -> CFCEBGCHDABAEHCEJCFF
path: ['CGEFIGIIGDBEGHICGAFF', 'CGEFIGIIGDDGEHICGAFF', 'CGEFIGIIGDDGEHCFGAFF', 'CGEFIGCBADDGEHCFGAFF', 'CGEFIGCBJCIGEHCFGAFF', 'CGEFIGCIAIIGEHCFGAFF', 'CGEFIGCIACBAEHCFGAFF', 'CGEFIGCIACBAEHCEJCFF', 'CGEFIGCHDABAEHCEJCFF', 'CFGBIGCHDABAEHCEJCFF', 'CFCEBGCHDABAEHCEJCFF']


In [ ]:
def beam_search(before, after, rules, width=100, ngram_n=1, max_steps=50):
    state_list = [(before, ngram(before, after, ngram_n), [before])]
    visited = set([before])

    for _ in range(max_steps):
        new_state_list = []

        for state, _, path in state_list:
            successors = find_successors(state, rules)
            for succ, _ in successors:  # successor = (next_state, rule)
                if succ in visited:
                    continue
                visited.add(succ)

                score = ngram(succ, after, ngram_n)
                new_path = path + [succ]

                # ゴール判定は文字列一致
                if succ == after:
                    return new_path

                new_state_list.append((succ, score, new_path))

        if not new_state_list:
            break

        # スコアが大きい順にソート（良いものを残す）
        state_list = sorted(new_state_list, key=lambda x: x[1], reverse=True)[:width]

    return None



path = beam_search(before, after, rules, width=100, ngram_n=1)
if path:
    print("Found path:", path)
else:
    print("No path found")





Found path: ['DCEFFGBFEHEGBBFJJEEE', 'BDIFFGBFEHEGBBFJJEEE', 'BDIFFGBFBEEGBBFJJEEE', 'BDIFDGBFBEEGBBFJJEEE', 'BDIFEEJFBEEGBBFJJEEE', 'BDIFEBGEBEEGBBFJJEEE', 'BDIFEBGEDIDGBBFJJEEE', 'BDIFEBGEDIHABBFJJEEE', 'BDIFEBGEBGCABBFJJEEE', 'BDIFEBGEBGCABJHAJEEE', 'BDIFEBGEBGCABJFFEEEE']


In [38]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 6.1 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [59]:
import pandas as pd
from tqdm import tqdm

def run_experiments(rules, alphabet, ngram_values, trials=50, length=20, step_num=20, width=100, max_steps=50):
    results = []
    for n in ngram_values:
        for t in tqdm(range(trials), desc=f"ngram={n}"):
            before, after, _ = sample_path(rules, alphabet, length=length, step_num=step_num, direction="backward")
            
            path = beam_search(before, after, rules, width=width, ngram_n=n, max_steps=max_steps)
            success = path is not None
            steps = len(path)-1 if success else None

            results.append({
                "ngram_n": n,
                "trial": t,
                "success": success,
                "steps": steps,
                "before": before,
                "after": after
            })
    return pd.DataFrame(results)

# 例: ngram_n = 1〜6 を比較
df = run_experiments(rules, alphabet, ngram_values=[1,2,3,4,5,6], trials=100)

# 成功率の集計
print(df.groupby("ngram_n")["success"].mean())

# ステップ数の平均（成功したケースのみ）
print(df[df["success"]].groupby("ngram_n")["steps"].mean())

# CSVに保存も可能
#df.to_csv("beam_search_ngram_results.csv", index=False)


ngram=6: 100%|██████████| 100/100 [00:11<00:00,  8.93it/s]

ngram_n
1    0.88
2    0.92
3    0.86
4    0.76
5    0.75
6    0.51
Name: success, dtype: float64
ngram_n
1    20.0
2    20.0
3    20.0
4    20.0
5    20.0
6    20.0
Name: steps, dtype: float64
